In [1]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OllamaEmbeddings
from langchain.vectorstores.pgvector import PGVector
from tqdm import tqdm
import psycopg2
import time
import os
from dotenv import load_dotenv

/home/snehal-modgil/anaconda3/envs/constitution-chatbot/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
loader = TextLoader("indian_constitution.txt", encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")

Split into 1129 chunks.


In [3]:
load_dotenv()

CONNECTION_STRING = os.environ.get("DATABASE_URL")
COLLECTION_NAME = "vectordb"

if not CONNECTION_STRING:
    raise ValueError("DATABASE_URL not found")

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print(f"🧠 Embedding {len(chunks)} chunks...")
start_embed = time.time()

texts = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]
embedding_list = [
    embeddings.embed_query(text) for text in tqdm(texts, desc="🔄 Embedding")
]

embed_duration = time.time() - start_embed
print(f"✅ Embedding done in {embed_duration:.2f} seconds ({embed_duration / len(embedding_list):.2f} sec/chunk)\n")

print(f"📥 Inserting {len(embedding_list)} vectors into pgvector...")
start_insert = time.time()

text_embedding_pairs = list(zip(texts, embedding_list))
vectorstore = PGVector.from_embeddings(
    text_embeddings=text_embedding_pairs,
    embedding=embeddings,
    metadatas=metadatas,
    collection_name=COLLECTION_NAME,
    connection_string=CONNECTION_STRING,
    pre_delete_collection=True,  # clears out any old/incompatible data under this collection name first
)

insert_duration = time.time() - start_insert
print(f"✅ Insert done in {insert_duration:.2f} seconds")


🧠 Embedding 1129 chunks...


🔄 Embedding: 100%|██████████| 1129/1129 [00:16<00:00, 67.83it/s]
/home/snehal-modgil/anaconda3/envs/constitution-chatbot/lib/python3.11/site-packages/langchain_community/vectorstores/pgvector.py:322: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata.Please note that filtering operators have been changed when using JSOB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create adb migration for your metadata column to be JSONB and update your queries to use the new operators. 
  warn_deprecated(
Collection not found


✅ Embedding done in 16.66 seconds (0.01 sec/chunk)

📥 Inserting 1129 vectors into pgvector...
✅ Insert done in 0.63 seconds


In [5]:
query = "What is Article 370?"
results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(doc.page_content[:300])
    print()


--- Result 1 ---
______________________________________________
1.Published with the Ministry of Law and Justice, (Legislative Department) notification
  No. G.S.R. 551 (E), dated the 5th August, 2019, Gazette of India, Extraordinary,
  Part II, Section 3, Sub-section (i).
                                         37

--- Result 2 ---
“370. All provisions of this Constitution, as amended from time to
         time, without any modifications or exceptions, shall apply to the State of
         Jammu and Kashmir notwithstanding anything contrary contained in
         article 152 or article 308 or any other article of this Constituti

--- Result 3 ---
______________________________________________
   In exercise of the powers conferred by clause (3) of article 370 read with clause (1) of
   article 370 of the Constitution of India, the President, on the recommendation of
   Parliament, is pleased to declare that, as from the 6 th August, 2019 all

--- Result 4 ---
terms thereof and to any a